# DPD — Revisión de archivos subidos

Subí dos Excel separados (uno por tabla) y este notebook:
1. Calcula DPD y arrears en **ambos modos** (cascade y join) por cuota.
2. **Identifica automáticamente** a cuál de los 18 casos del catálogo se parece cada contrato.
3. Devuelve la tabla del schedule con 5 columnas nuevas: `CASCADE_DPD`, `CASCADE_ARREARS`, `JOIN_DPD`, `JOIN_ARREARS`, `CASO`.
4. Exporta un Excel con dos hojas (`schedule_con_dpd` y `resumen_por_contrato`).

**Editá los paths y `CALC_DATE` en la celda de configuración**, después corré las celdas en orden.

In [30]:
import os
import sys
from datetime import date
from decimal import Decimal

import pandas as pd

sys.path.insert(0, os.path.abspath("."))
from dpd.config import RunConfig
from dpd.modes import join_installment, cascade_fifo

# === Parámetros — editar acá ===
SCHEDULE_PATH = "tests/Days Past Due.xlsx"   # Excel con la tabla scheduled_payments_installments
PT_PATH = "tests/Days Past Due.xlsx"          # Excel con la tabla payment_tape

# Si los dos paths apuntan al mismo archivo, asumimos que son las hojas
# 'scheduled_payments_installments' y 'Payment_Tape' del mismo workbook.
# Si son archivos separados, cada uno debe tener una sola hoja con los datos.
SCHEDULE_SHEET = None     # None = primera hoja; o nombre exacto si querés especificar
PT_SHEET = None

CALC_DATE = date(2026, 10, 3)   # FECHA DE CORTE — manual obligatoria
PARTIAL_COUNTS = False           # solo afecta cascade

OUT_PATH = "resultado_revision.xlsx"

print(f"Schedule: {SCHEDULE_PATH}")
print(f"PT:       {PT_PATH}")
print(f"Corte:    {CALC_DATE}")

Schedule: tests/Days Past Due.xlsx
PT:       tests/Days Past Due.xlsx
Corte:    2026-10-03


In [31]:
# === Cargar y sanitizar ===
def _load_sheet(path, sheet_name, fallback_names):
    """Carga la hoja indicada o, si sheet_name es None, intenta los fallbacks o la primera."""
    xls = pd.ExcelFile(path)
    if sheet_name and sheet_name in xls.sheet_names:
        return pd.read_excel(xls, sheet_name=sheet_name)
    for fb in fallback_names:
        if fb in xls.sheet_names:
            return pd.read_excel(xls, sheet_name=fb)
    return pd.read_excel(xls, sheet_name=xls.sheet_names[0])


sched_df = _load_sheet(
    SCHEDULE_PATH, SCHEDULE_SHEET,
    ["scheduled_payments_installments", "schedule", "Schedule"],
)
pay_df = _load_sheet(
    PT_PATH, PT_SHEET,
    ["Payment_Tape", "payment_tape", "PT"],
)

# Sanitización — mover Unnamed:NN a borrower_installment_reference si está vacía
if "borrower_installment_reference" in pay_df.columns and pay_df["borrower_installment_reference"].isna().all():
    fb = next((c for c in pay_df.columns if c.startswith("Unnamed:") and pay_df[c].notna().any()), None)
    if fb:
        print(f"⚠ pay_df.borrower_installment_reference vacía — uso '{fb}' como fallback.")
        pay_df["borrower_installment_reference"] = pay_df[fb]

# Tipos
sched_df["date"] = pd.to_datetime(sched_df["date"]).dt.date
pay_df["payment_date"] = pd.to_datetime(pay_df["payment_date"]).dt.date

def _ref_to_str(s):
    if pd.api.types.is_float_dtype(s):
        return s.where(s.isna(), s.astype("Int64")).astype(str)
    return s.astype(str)

sched_df["borrower_installment_reference"] = _ref_to_str(sched_df["borrower_installment_reference"])
pay_df["borrower_installment_reference"] = _ref_to_str(pay_df["borrower_installment_reference"])

# id sintético si no viene del Excel
sched_df = sched_df.reset_index(drop=True)
if "id" not in sched_df.columns or sched_df["id"].isna().all():
    sched_df["id"] = sched_df.index + 1

# === Auto-relleno de buckets (opción 6A) ===
# Si la suma de buckets es 0 pero gross_amount > 0, asumimos principal_amount = gross_amount.
# Esto permite que cascade funcione aunque el Excel no traiga el desglose.
BUCKET_COLS = ["principal_amount", "interest_amount", "guarantee_amount", "tax_amount", "fee_amount"]
for col in BUCKET_COLS:
    if col not in sched_df.columns:
        sched_df[col] = 0
sched_df[BUCKET_COLS] = sched_df[BUCKET_COLS].fillna(0)

bucket_sum = sched_df[BUCKET_COLS].sum(axis=1)
need_fallback = (bucket_sum == 0) & (sched_df["gross_amount"].fillna(0) > 0)
n_filled = int(need_fallback.sum())
if n_filled > 0:
    sched_df.loc[need_fallback, "principal_amount"] = sched_df.loc[need_fallback, "gross_amount"]
    print(f"⚠ {n_filled} cuota(s) sin buckets → asumimos principal_amount = gross_amount.")

print(f"Schedule: {len(sched_df)} cuotas | {sched_df['borrower_contract_id'].nunique()} contratos")
print(f"PT:       {len(pay_df)} pagos     | {pay_df['borrower_contract_id'].nunique()} contratos")

Schedule: 180 cuotas | 18 contratos
PT:       73 pagos     | 16 contratos


In [32]:
# === Clasificador de casos (1-18) ===
# Cada regla detecta un patrón específico. Permitimos múltiples matches (B2):
# si los datos son indistinguibles entre casos (ej. 3 y 17), se devuelven todos.
# Si no calza con ninguno → ['Sin clasificar'].
#
# Caso 14 (Reestructurado) NO se puede detectar sin metadata extra (status/version).

def _enrich_contract(sched_c, pay_c, calc_date):
    """sched ordenado por fecha + total_paid_by_ref, n_pagos_ref, overdue."""
    sched_c = sched_c.copy().sort_values("date").reset_index(drop=True)
    if len(pay_c) > 0:
        agg = (pay_c.groupby("borrower_installment_reference", as_index=False)
                    .agg(total_paid_by_ref=("total_payment", "sum"),
                         n_pagos_ref=("total_payment", "count")))
    else:
        agg = pd.DataFrame(columns=["borrower_installment_reference", "total_paid_by_ref", "n_pagos_ref"])
    sched_c = sched_c.merge(agg, on="borrower_installment_reference", how="left")
    sched_c["total_paid_by_ref"] = sched_c["total_paid_by_ref"].fillna(0).astype(float)
    sched_c["n_pagos_ref"] = sched_c["n_pagos_ref"].fillna(0).astype(int)
    sched_c["overdue"] = sched_c["date"] <= calc_date
    return sched_c


def _has_late_pay(sched_c, pay_c):
    if len(pay_c) == 0:
        return False
    merged = pay_c.merge(
        sched_c[["borrower_installment_reference", "date"]],
        on="borrower_installment_reference", how="left",
    )
    return ((merged["payment_date"] > merged["date"]) & merged["date"].notna()).any()


def _has_early_pay(sched_c, pay_c):
    if len(pay_c) == 0:
        return False
    merged = pay_c.merge(
        sched_c[["borrower_installment_reference", "date"]],
        on="borrower_installment_reference", how="left",
    )
    return ((merged["payment_date"] < merged["date"]) & merged["date"].notna()).any()


def classify_contract(sched_c, pay_c, calc_date):
    """Lista de casos (1-18) que matchean. Vacío → ['Sin clasificar']."""
    n_pagos = len(pay_c)
    matches = []

    enriched = _enrich_contract(sched_c, pay_c, calc_date)
    overdue = enriched[enriched["overdue"]].reset_index(drop=True)
    n_overdue = len(overdue)

    if n_overdue == 0:
        return ["Sin clasificar"]

    # Casos 3 y 17: sin pagos. Indistinguibles por datos.
    if n_pagos == 0:
        return [3, 17]

    paid_exact = (overdue["total_paid_by_ref"] - overdue["gross_amount"]).abs() < 0.001
    partial = (overdue["total_paid_by_ref"] > 0) & (overdue["total_paid_by_ref"] + 0.001 < overdue["gross_amount"])
    impaga = overdue["total_paid_by_ref"] == 0
    sobrepagada = overdue["total_paid_by_ref"] > overdue["gross_amount"] + 0.001
    covered = overdue["total_paid_by_ref"] >= overdue["gross_amount"] - 0.001  # paid_exact OR sobrepagada

    n_paid = paid_exact.sum()
    n_partial = partial.sum()
    n_impaga = impaga.sum()
    n_sobre = sobrepagada.sum()
    n_covered = covered.sum()

    has_late = _has_late_pay(enriched, pay_c)
    has_early = _has_early_pay(enriched, pay_c)
    has_pay_on_corte = (pay_c["payment_date"] == calc_date).any()

    diff = overdue["gross_amount"] - overdue["total_paid_by_ref"]
    has_centavos = ((diff > 0.001) & (diff < 1.0)).any()

    # Cuota con 2 pagos en el mismo mes calendario que sumen gross
    has_doble_pago_mismo_mes = False
    for ref in overdue["borrower_installment_reference"].unique():
        ref_pays = pay_c[pay_c["borrower_installment_reference"] == ref]
        if len(ref_pays) == 2:
            cuota_gross = float(overdue.loc[overdue["borrower_installment_reference"] == ref, "gross_amount"].iloc[0])
            if abs(ref_pays["total_payment"].sum() - cuota_gross) < 0.01:
                d1, d2 = list(ref_pays["payment_date"])
                if d1.year == d2.year and d1.month == d2.month:
                    has_doble_pago_mismo_mes = True
                    break

    # ─── Reglas exclusivas ──────────────────────────────────────────────────

    # Caso 1: Happy path estricto
    if (n_paid == n_overdue and not has_late and not has_early and n_sobre == 0):
        matches.append(1)

    # Caso 2: Última cuota vencida es la única parcial
    if (n_overdue >= 2 and n_partial == 1 and n_impaga == 0 and n_sobre == 0
            and partial.iloc[-1] and paid_exact.iloc[:-1].all()):
        matches.append(2)

    # Caso 4: Cubiertas con pago tardío, una cuota con sobrepago modesto OK (intereses moratorios).
    # Excluye caso 10 (doble pago mismo mes) y caso 12 (pago el día del corte).
    if (n_covered == n_overdue and has_late and not has_early
            and (overdue["n_pagos_ref"] <= 1).all()
            and not has_doble_pago_mismo_mes
            and not has_pay_on_corte):
        matches.append(4)

    # Caso 5: Una cuota intermedia parcial sin recuperar (ni la última, ni centavos)
    if (n_overdue >= 3 and n_partial == 1 and n_impaga == 0 and n_sobre == 0
            and not partial.iloc[-1] and (paid_exact | partial).all() and not has_centavos):
        matches.append(5)

    # Caso 6: Cubiertas con pago anticipado, sin tardío
    if (n_paid == n_overdue and has_early and not has_late and n_sobre == 0):
        matches.append(6)

    # Caso 7: Sobrepago modesto en exactamente una cuota (1.001 ≤ ratio ≤ 1.4),
    # sin tardío, sin temprano, sin pago en corte
    if n_sobre == 1 and n_partial == 0 and n_impaga == 0:
        ratio = (overdue["total_paid_by_ref"] / overdue["gross_amount"]).max()
        if 1.001 <= ratio <= 1.4 and not has_late and not has_early and not has_pay_on_corte:
            matches.append(7)

    # Caso 8: 2+ cuotas con pago parcial
    if n_partial >= 2:
        matches.append(8)

    # Caso 9: Sobrepago ~1.5x en una cuota Y la siguiente cubre menos que gross
    for i in range(n_overdue - 1):
        gross_i = overdue["gross_amount"].iloc[i]
        if gross_i > 0:
            rat_i = overdue["total_paid_by_ref"].iloc[i] / gross_i
            if 1.4 <= rat_i <= 1.6:
                next_paid = overdue["total_paid_by_ref"].iloc[i + 1]
                next_gross = overdue["gross_amount"].iloc[i + 1]
                if next_paid + 0.001 < next_gross:
                    matches.append(9)
                    break

    # Caso 10: Cuota con 2 pagos del mismo mes que sumen gross
    if has_doble_pago_mismo_mes:
        matches.append(10)

    # Caso 11: Una sola cuota pagada (la primera), resto impagas
    if n_paid == 1 and n_impaga >= 2 and n_partial == 0 and paid_exact.iloc[0]:
        matches.append(11)

    # Caso 12: Algún pago el día del corte
    if has_pay_on_corte:
        matches.append(12)

    # Caso 13: Diferencia de centavos en alguna cuota
    if has_centavos:
        matches.append(13)

    # Caso 14: Reestructurado — no detectable sin metadata. Skip.

    # Caso 15: Cuota con sobrepago ≥ 1.9x (duplicado)
    ratios = overdue["total_paid_by_ref"] / overdue["gross_amount"].replace(0, 1)
    if (ratios >= 1.9).any():
        matches.append(15)

    # Caso 16: Primera cuota vencida impaga, resto cubiertas exactas
    if (n_overdue >= 2 and impaga.iloc[0] and paid_exact.iloc[1:].all()
            and n_partial == 0 and n_sobre == 0):
        matches.append(16)

    # Caso 18: Pagos consecutivos al inicio + impagas consecutivas al final, sin tardío
    if (n_paid >= 2 and n_impaga >= 1 and n_partial == 0 and n_sobre == 0 and not has_late):
        first_unpaid_idx = next((i for i, p in enumerate(paid_exact.values) if not p), None)
        if first_unpaid_idx is not None and first_unpaid_idx >= 2:
            before = paid_exact.iloc[:first_unpaid_idx]
            after_impaga = impaga.iloc[first_unpaid_idx:]
            if before.all() and after_impaga.all():
                matches.append(18)

    return matches if matches else ["Sin clasificar"]


def caso_str(matches):
    if not matches:
        return "Sin clasificar"
    return ", ".join(str(m) for m in matches)


print("Clasificador definido.")

Clasificador definido.


In [33]:
# === Cómputo: validación + cascade + join + clasificación por contrato ===
def installments_from_df(df):
    """Filas inválidas (date NaT, gross NaN o ≤0) se omiten — quedarán con NaN en CASCADE/JOIN
    después del merge final, y la columna MATCH=False explica que el contrato tiene errores."""
    valid = df[df["date"].notna() & (df["gross_amount"].fillna(0) > 0)]
    return [
        {
            "id": int(r["id"]),
            "borrower_contract_id": r["borrower_contract_id"],
            "borrower_installment_reference": r["borrower_installment_reference"],
            "installment_date": r["date"],
            "gross_amount": r.get("gross_amount", 0),
            "guarantee_amount": r.get("guarantee_amount", 0),
            "principal_amount": r.get("principal_amount", 0),
            "interest_amount": r.get("interest_amount", 0),
            "tax_amount": r.get("tax_amount", 0),
            "fee_amount": r.get("fee_amount", 0),
        }
        for _, r in valid.iterrows()
    ]


def payments_from_df(df):
    """Pagos con payment_date NaT o total_payment ≤0 se omiten del cómputo (pero quedan en
    el summary y el cómputo de errores)."""
    valid = df[df["payment_date"].notna() & (df["total_payment"].fillna(0) > 0)]
    return [
        {
            "borrower_contract_id": r["borrower_contract_id"],
            "borrower_installment_reference": r.get("borrower_installment_reference"),
            "payment_date": r["payment_date"],
            "total_payment": r.get("total_payment", 0),
        }
        for _, r in valid.iterrows()
    ]


def validate_contract(sched_c, pay_c):
    """Detecta errores de datos por contrato. Devuelve 'OK' o 'err1; err2; ...'.
    No descarta filas — solo reporta."""
    errs = []

    # Schedule
    n_gross_bad = (sched_c["gross_amount"].isna() | (sched_c["gross_amount"].fillna(0) <= 0)).sum()
    if n_gross_bad > 0:
        errs.append(f"{n_gross_bad} cuota(s) con gross_amount inválido")

    n_date_bad = sched_c["date"].isna().sum()
    if n_date_bad > 0:
        errs.append(f"{n_date_bad} cuota(s) sin fecha")

    refs_sched = sched_c["borrower_installment_reference"].dropna().astype(str)
    dup_refs = refs_sched[refs_sched.duplicated()].unique()
    if len(dup_refs) > 0:
        errs.append(f"refs duplicadas en schedule: {list(dup_refs)[:3]}")

    # Payment_Tape
    if len(pay_c) > 0:
        n_pd_bad = pay_c["payment_date"].isna().sum()
        if n_pd_bad > 0:
            errs.append(f"{n_pd_bad} pago(s) sin payment_date")

        n_tp_bad = (pay_c["total_payment"].isna() | (pay_c["total_payment"].fillna(0) <= 0)).sum()
        if n_tp_bad > 0:
            errs.append(f"{n_tp_bad} pago(s) con total_payment inválido")

        sched_refs_set = set(refs_sched)
        pay_refs = pay_c["borrower_installment_reference"].dropna().astype(str)
        unknown = [r for r in pay_refs if r and r != "nan" and r not in sched_refs_set]
        if unknown:
            errs.append(f"{len(unknown)} pago(s) con ref no encontrada en schedule")

    return "OK" if not errs else "; ".join(errs)


# === Validación por contrato ===
errores_por_contrato = {}
for cid, sched_c in sched_df.groupby("borrower_contract_id"):
    pay_c = pay_df[pay_df["borrower_contract_id"] == cid]
    errores_por_contrato[cid] = validate_contract(sched_c, pay_c)

n_con_errores = sum(1 for v in errores_por_contrato.values() if v != "OK")
if n_con_errores > 0:
    print(f"⚠ {n_con_errores}/{len(errores_por_contrato)} contratos con errores. Procesando todos igual.")
    for cid, err in errores_por_contrato.items():
        if err != "OK":
            print(f"   {cid}: {err}")
else:
    print(f"✓ {len(errores_por_contrato)} contratos sin errores de datos.")

# === Cómputo en ambos modos ===
insts = installments_from_df(sched_df)
pays = payments_from_df(pay_df)

cfg_c = RunConfig(company_id=1, company_code="*", mode="cascade",
                  partial_payment_counts=PARTIAL_COUNTS, calculation_date=CALC_DATE)
cfg_j = RunConfig(company_id=1, company_code="*", mode="join",
                  partial_payment_counts=False, calculation_date=CALC_DATE)

c_results = list(cascade_fifo.compute_from_data(insts, pays, cfg_c))
j_results = list(join_installment.compute_from_data(insts, pays, cfg_j))

cdf = pd.DataFrame(c_results).rename(columns={
    "dpd_current": "CASCADE_DPD", "amount_in_arrears": "CASCADE_ARREARS",
})
jdf = pd.DataFrame(j_results).rename(columns={
    "dpd_current": "JOIN_DPD", "amount_in_arrears": "JOIN_ARREARS",
})
if not cdf.empty:
    cdf["CASCADE_ARREARS"] = cdf["CASCADE_ARREARS"].astype(float)
if not jdf.empty:
    jdf["JOIN_ARREARS"] = jdf["JOIN_ARREARS"].astype(float)

# === Clasificar por contrato ===
caso_por_contrato = {}
for cid, sched_c in sched_df.groupby("borrower_contract_id"):
    pay_c = pay_df[pay_df["borrower_contract_id"] == cid]
    try:
        matches = classify_contract(sched_c, pay_c, CALC_DATE)
        caso_por_contrato[cid] = caso_str(matches)
    except Exception as e:
        # Si el clasificador revienta (ej. data muy rota), marcamos sin clasificar
        caso_por_contrato[cid] = "Sin clasificar"

# === Tabla final: schedule + columnas calculadas ===
result_df = sched_df.merge(cdf, on="id", how="left").merge(jdf, on="id", how="left")
result_df["CASE"] = result_df["borrower_contract_id"].map(caso_por_contrato)
# MATCH: True si no hay errores de datos en ese contrato, False si los hay.
result_df["MATCH"] = result_df["borrower_contract_id"].map(errores_por_contrato).eq("OK")

# Cuando DPD = 0 la cuota no está en mora — el "arrears" que devuelve el motor en ese caso
# es el saldo pendiente futuro de la cuota, no una mora. Lo ponemos en 0 para que la tabla
# detallada refleje únicamente mora real.
result_df.loc[result_df["CASCADE_DPD"] == 0, "CASCADE_ARREARS"] = 0.0
result_df.loc[result_df["JOIN_DPD"] == 0, "JOIN_ARREARS"] = 0.0

# MAX_*_DPD: pico de mora alcanzado en el contrato (máximo entre las cuotas vencidas a la
# fecha de corte). Se replica en cada fila del contrato para verlo en la vista detallada.
result_df["MAX_CASCADE_DPD"] = result_df.groupby("borrower_contract_id")["CASCADE_DPD"].transform("max")
result_df["MAX_JOIN_DPD"] = result_df.groupby("borrower_contract_id")["JOIN_DPD"].transform("max")

new_cols = ["CASCADE_DPD", "CASCADE_ARREARS", "MAX_CASCADE_DPD",
            "JOIN_DPD", "JOIN_ARREARS", "MAX_JOIN_DPD",
            "CASE", "MATCH"]
old_cols = [c for c in result_df.columns if c not in new_cols]
result_df = result_df[old_cols + new_cols]

n_calc = result_df["CASCADE_DPD"].notna().sum()
print(f"\nCuotas calculadas: {n_calc}/{len(result_df)} | DPD cascade>0: {(result_df['CASCADE_DPD'] > 0).sum()} | DPD join>0: {(result_df['JOIN_DPD'] > 0).sum()}")
if n_calc < len(result_df):
    print(f"  ({len(result_df) - n_calc} cuotas omitidas del cómputo por gross/date inválidos)")

✓ 18 contratos sin errores de datos.

Cuotas calculadas: 180/180 | DPD cascade>0: 24 | DPD join>0: 25


/var/folders/kk/cyrcsdy128dcqq7mxn7khzf80000gn/T/ipykernel_90066/2450968626.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sched_c["total_paid_by_ref"] = sched_c["total_paid_by_ref"].fillna(0).astype(float)
/var/folders/kk/cyrcsdy128dcqq7mxn7khzf80000gn/T/ipykernel_90066/2450968626.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sched_c["n_pagos_ref"] = sched_c["n_pagos_ref"].fillna(0).astype(int)
/var/folders/kk/cyrcsdy128dcqq7mxn7khzf80000gn/T/ipykernel_90066/2450968626.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill

In [34]:
# === Resumen por contrato (una fila por contrato) ===
# Cada total de arrears suma sólo cuotas con DPD > 0 EN ESE MODO.
# Así cascade_arrears_total refleja la mora real de cascada (cuotas vencidas no cubiertas
# por el pool FIFO) y join_arrears_total la mora del modo join — sin mezclar saldos
# futuros que no están realmente en mora.
arrears_c = result_df.where(result_df["CASCADE_DPD"] > 0)["CASCADE_ARREARS"]
arrears_j = result_df.where(result_df["JOIN_DPD"] > 0)["JOIN_ARREARS"]
result_df["_CASCADE_ARREARS_OVERDUE"] = arrears_c
result_df["_JOIN_ARREARS_OVERDUE"] = arrears_j

summary = (
    result_df.groupby("borrower_contract_id", as_index=False)
             .agg(
                 cuotas=("id", "count"),
                 cascade_dpd_max=("CASCADE_DPD", "max"),
                 cascade_arrears_total=("_CASCADE_ARREARS_OVERDUE", "sum"),
                 join_dpd_max=("JOIN_DPD", "max"),
                 join_arrears_total=("_JOIN_ARREARS_OVERDUE", "sum"),
                 case=("CASE", "first"),
                 match=("MATCH", "first"),
             )
             .sort_values("cascade_dpd_max", ascending=False)
)

result_df = result_df.drop(columns=["_CASCADE_ARREARS_OVERDUE", "_JOIN_ARREARS_OVERDUE"])
summary

,borrower_contract_id,cuotas,cascade_dpd_max,cascade_arrears_total,join_dpd_max,join_arrears_total,case,match
16,e7b3f1c8-2a64-b5d9-c1f7-6ab4d8c1f2b5,10,124,1322530.0,124,1322530.0,"3, 17",True
14,c4d9e1f7-2b86-a3c5-84d1-15a7c8b9e334,10,124,1322530.0,124,1322530.0,"3, 17",True
0,0e7f3a8b-1c64-b9d5-c2a8-04e1c3b8d456,10,94,1058024.0,94,1058024.0,11,True
4,4a8c2b9e-7f31-d6c4-b5e9-58c3a7d1e924,10,32,529012.0,32,529012.0,18,True
6,5a1f8d6e-2c93-b4a7-c5d9-71e3b9c2d847,10,32,396759.0,63,396759.0,8,True
1,1a5e8b3c-9d27-f4a6-b8e1-26d9f5c3a278,10,2,0.5,63,0.5,13,True
15,d3e9b4a1-6f57-c8b2-a4f1-82a5c1d9e673,10,2,132253.0,2,264506.0,9,True
12,a8f3c92d-4e71-b8a5-91c2-09efb6e1d772,10,2,132253.0,2,132253.0,2,True
17,f1a7c3e8-5d94-b6f2-83a1-47d9e2b1c948,10,2,132253.0,63,132253.0,5,True
8,6c2d7f1b-5e94-a3c8-f6d2-37e1a4b8c389,10,2,264506.0,2,264506.0,Sin clasificar,True


In [35]:
# === Tabla detallada (schedule + 6 columnas) ===
from IPython.display import display

with pd.option_context("display.max_rows", 50):
    display(result_df)

,id,company_code,contract_type,borrower_contract_id,atom_contract_id,borrower_installment_reference,date,gross_amount,net_amount,fee_amount,...,NPV,days past due,CASCADE_DPD,CASCADE_ARREARS,MAX_CASCADE_DPD,JOIN_DPD,JOIN_ARREARS,MAX_JOIN_DPD,CASE,MATCH
0,1,1,NaN,6603dce6-6b76-c0dc-96c1-08ddb5d9c44e,xyz123,1,2026-06-01,264506,NaN,0.0,...,NaN,NaN,0,0.0,0,0,0.0,0,1,True
1,2,1,NaN,6603dce6-6b76-c0dc-96c1-08ddb5d9c44e,xyz123,2,2026-07-01,264506,NaN,0.0,...,NaN,NaN,0,0.0,0,0,0.0,0,1,True
2,3,1,NaN,6603dce6-6b76-c0dc-96c1-08ddb5d9c44e,xyz123,3,2026-08-01,264506,NaN,0.0,...,NaN,NaN,0,0.0,0,0,0.0,0,1,True
3,4,1,NaN,6603dce6-6b76-c0dc-96c1-08ddb5d9c44e,xyz123,4,2026-09-01,264506,NaN,0.0,...,NaN,NaN,0,0.0,0,0,0.0,0,1,True
4,5,1,NaN,6603dce6-6b76-c0dc-96c1-08ddb5d9c44e,xyz123,5,2026-10-01,264506,NaN,0.0,...,NaN,NaN,0,0.0,0,0,0.0,0,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,176,1,NaN,4a8c2b9e-7f31-d6c4-b5e9-58c3a7d1e924,xyz140,6,2026-11-01,264506,NaN,0.0,...,NaN,NaN,0,0.0,32,0,0.0,32,18,True
176,177,1,NaN,4a8c2b9e-7f31-d6c4-b5e9-58c3a7d1e924,xyz140,7,2026-12-01,264506,NaN,0.0,...,NaN,NaN,0,0.0,32,0,0.0,32,18,True
177,178,1,NaN,4a8c2b9e-7f31-d6c4-b5e9-58c3a7d1e924,xyz140,8,2027-01-01,264506,NaN,0.0,...,NaN,NaN,0,0.0,32,0,0.0,32,18,True
178,179,1,NaN,4a8c2b9e-7f31-d6c4-b5e9-58c3a7d1e924,xyz140,9,2027-02-01,264506,NaN,0.0,...,NaN,NaN,0,0.0,32,0,0.0,32,18,True


In [36]:
# === Export a Excel con dos hojas ===
with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    result_df.to_excel(writer, sheet_name="schedule_con_dpd", index=False)
    summary.to_excel(writer, sheet_name="resumen_por_contrato", index=False)

print(f"Exportado: {os.path.abspath(OUT_PATH)}")
print(f"  - schedule_con_dpd:    {len(result_df)} filas")
print(f"  - resumen_por_contrato: {len(summary)} filas")

Exportado: /Users/d.salamanca/Vaas/DaysPostDue/resultado_revision.xlsx
  - schedule_con_dpd:    180 filas
  - resumen_por_contrato: 18 filas
